<a href="https://colab.research.google.com/github/Marianela-Fontana/dmeyf2026/blob/main/ENS_6300_6302.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
require("data.table")

PARAM <- list()
PARAM$experimento <- "ENS_6300_6302"
PARAM$kaggle <- list()
PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(600, 1200, by = 50)   # ajustá a tu límite diario de envíos

# 1) leo las predicciones de los dos experimentos (revisá las rutas con ls)
archivos <- c(
  "buckets/b1/exp/WF6300/prediccion.txt",
  "buckets/b1/exp/WF6302/prediccion.txt"
)
tb_todas <- rbindlist(lapply(archivos, fread))

# 2) ensemble: promedio de probabilidad por cliente
tb_prediccion <- tb_todas[, list(prob = mean(prob)), by = numero_de_cliente]
setorder(tb_prediccion, -prob)

# 3) mismo bucle que el profe
dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle, sep = ","
  )

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'ensemble 6300+6302 envios=", envios, "'")

  linea <- paste(comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern = TRUE)
  cat(salida, "\n")
}